# Temperature Recommendation - Energy-Aware Transformer

Trains the temperature recommendation Transformer with the new energy-aware comfort-constrained target.

Artifacts are saved back to Google Drive under `AIModelsAndAlgorithms/TempretureRecomendation/transformer`.


In [1]:
from google.colab import drive
from pathlib import Path
import os
import subprocess

# 1) Mount Google Drive for datasets and saved training outputs.
drive.mount('/content/drive')

# 2) Edit these two values for your GitHub repo.
GITHUB_REPO_URL = 'https://github.com/abdullahtapanci/VisualizationApp.git'
GITHUB_BRANCH = 'main'  # change if you use another branch

# 3) Drive folders. Datasets stay in Drive; code is cloned from GitHub.
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/VisualizationApp')
DRIVE_DATA_DIR = DRIVE_PROJECT_DIR / 'Data'
DRIVE_OUTPUT_ROOT = DRIVE_PROJECT_DIR
PROJECT_DIR = Path('/content/VisualizationApp')

if 'YOUR_USERNAME' in GITHUB_REPO_URL:
    raise ValueError('Edit GITHUB_REPO_URL to your real GitHub repository URL, then rerun this cell.')
if not DRIVE_DATA_DIR.exists():
    raise FileNotFoundError(f'Drive dataset folder was not found: {DRIVE_DATA_DIR}')

# 4) Clone or update code from GitHub.
if PROJECT_DIR.exists():
    print('Repository already exists. Pulling latest changes...')
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'checkout', GITHUB_BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', 'origin', GITHUB_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', GITHUB_BRANCH, GITHUB_REPO_URL, str(PROJECT_DIR)], check=True)

# 5) Link Drive datasets into the cloned repo so existing scripts read Data/*.csv.
LOCAL_DATA_DIR = PROJECT_DIR / 'Data'
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
for filename in ['PIRSensorData.csv', 'hotelReservationData.csv', 'lightningData.csv', 'temperatureData.csv', 'WheatherDataAntalya.csv']:
    src = DRIVE_DATA_DIR / filename
    dst = LOCAL_DATA_DIR / filename
    if src.exists():
        if dst.exists() or dst.is_symlink():
            dst.unlink()
        os.symlink(src, dst)
        print('linked', dst, '->', src)
    else:
        print('not found in Drive, skipping:', src)

DATA_DIR = LOCAL_DATA_DIR
print('PROJECT_DIR =', PROJECT_DIR)
print('DATA_DIR =', DATA_DIR)
print('DRIVE_OUTPUT_ROOT =', DRIVE_OUTPUT_ROOT)


Mounted at /content/drive
linked /content/VisualizationApp/Data/PIRSensorData.csv -> /content/drive/MyDrive/VisualizationApp/Data/PIRSensorData.csv
linked /content/VisualizationApp/Data/hotelReservationData.csv -> /content/drive/MyDrive/VisualizationApp/Data/hotelReservationData.csv
linked /content/VisualizationApp/Data/lightningData.csv -> /content/drive/MyDrive/VisualizationApp/Data/lightningData.csv
linked /content/VisualizationApp/Data/temperatureData.csv -> /content/drive/MyDrive/VisualizationApp/Data/temperatureData.csv
linked /content/VisualizationApp/Data/WheatherDataAntalya.csv -> /content/drive/MyDrive/VisualizationApp/Data/WheatherDataAntalya.csv
PROJECT_DIR = /content/VisualizationApp
DATA_DIR = /content/VisualizationApp/Data
DRIVE_OUTPUT_ROOT = /content/drive/MyDrive/VisualizationApp


In [2]:
# Optional: if your new dataset files are in another Drive folder, set NEW_DATA_DIR and run this cell.
# The files will be linked/copied through DRIVE_DATA_DIR and then used by the cloned repo.
# Example: NEW_DATA_DIR = Path('/content/drive/MyDrive/NewVisualizationAppData')
NEW_DATA_DIR = None

if NEW_DATA_DIR is not None:
    for filename in ['PIRSensorData.csv', 'hotelReservationData.csv', 'lightningData.csv', 'temperatureData.csv', 'WheatherDataAntalya.csv']:
        src = Path(NEW_DATA_DIR) / filename
        dst = DRIVE_DATA_DIR / filename
        if src.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            !cp "{src}" "{dst}"
            print('copied', src, '->', dst)
        else:
            print('missing in NEW_DATA_DIR:', src)


In [3]:
!pip install -q pandas numpy scikit-learn joblib matplotlib torch


In [4]:
assert (DATA_DIR / 'temperatureData.csv').exists(), 'Missing temperatureData.csv'
OUTPUT_DIR = DRIVE_OUTPUT_ROOT / 'AIModelsAndAlgorithms/TempretureRecomendation/transformer'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('OUTPUT_DIR =', OUTPUT_DIR)


OUTPUT_DIR = /content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/TempretureRecomendation/transformer


## Smoke Test
Run this first to verify paths and dependencies without training on the full dataset.


In [5]:
%cd {PROJECT_DIR}
!python AIModelsAndAlgorithms/TempretureRecomendation/transformer/train_energy_aware_transformer.py --max-rows 500000 --max-sequences 50000 --sequence-length 12 --stride 12 --epochs 2 --batch-size 256 --output-dir "{OUTPUT_DIR}"


/content/VisualizationApp
/content/VisualizationApp/AIModelsAndAlgorithms/TempretureRecomendation/transformer/train_energy_aware_transformer.py:252: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
epoch 01 train_loss=0.04872 val_loss=0.02898
epoch 02 train_loss=0.02767 val_loss=0.01551
rows: 500,000
sequences: 41,600
train sequences: 33,280
val sequences: 4,160
test sequences: 4,160
algorithm: TransformerEncoder
target strategy: energy-aware comfort-constrained setpoint
MAE: 0.674
RMSE: 1.721
R2: 0.810
current energy proxy W-sum: 2313185
target energy proxy W-sum: 1808720
model energy proxy W-sum: 1854240
target saving vs current: 21.81%
model saving vs current: 19.84%
target mean comfort gap C: 4.439
model mean comfort gap C: 4.212

saved /content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/TempretureRecomendation/transformer/tempreture_r

## Full Training
Run this cell when you are ready. It may take a long time on the full dataset.


In [6]:
%cd {PROJECT_DIR}
!python AIModelsAndAlgorithms/TempretureRecomendation/transformer/train_energy_aware_transformer.py --max-rows 2000000 --max-sequences 300000 --sequence-length 12 --stride 6 --epochs 20 --batch-size 256 --output-dir "{OUTPUT_DIR}"


/content/VisualizationApp
/content/VisualizationApp/AIModelsAndAlgorithms/TempretureRecomendation/transformer/train_energy_aware_transformer.py:252: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
epoch 01 train_loss=0.01909 val_loss=0.00818
epoch 02 train_loss=0.01021 val_loss=0.00903
epoch 03 train_loss=0.00893 val_loss=0.01000
epoch 04 train_loss=0.00733 val_loss=0.00768
epoch 05 train_loss=0.00686 val_loss=0.00806
epoch 06 train_loss=0.00609 val_loss=0.00585
epoch 07 train_loss=0.00600 val_loss=0.00410
epoch 08 train_loss=0.00604 val_loss=0.00617
epoch 09 train_loss=0.00540 val_loss=0.00465
epoch 10 train_loss=0.00528 val_loss=0.00396
epoch 11 train_loss=0.00523 val_loss=0.00660
epoch 12 train_loss=0.00512 val_loss=0.00955
epoch 13 train_loss=0.00485 val_loss=0.00833
epoch 14 train_loss=0.00534 val_loss=0.01321
epoch 15 train_loss=0.00489 val_l

## Check Saved Artifacts


In [7]:
print('Saved files:')
for path in sorted(OUTPUT_DIR.glob('*')):
    print(path)


Saved files:
/content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/TempretureRecomendation/transformer/tempreture_recomendation_transformer.pt
/content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/TempretureRecomendation/transformer/tempreture_recomendation_transformer_metadata.json
/content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/TempretureRecomendation/transformer/tempreture_recomendation_transformer_report.txt
/content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/TempretureRecomendation/transformer/tempreture_recomendation_transformer_sample_predictions.csv
